In [32]:
from langchain.agents import create_agent

from IPython.display import Markdown

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

from langchain_ollama import ChatOllama

from langchain_community.llms import Ollama

from langchain.messages import HumanMessage,SystemMessage,AIMessage

from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call

from IPython.display import Markdown

from langgraph.checkpoint.memory import MemorySaver


In [33]:
load_dotenv(override=True)

True

In [34]:
llm = ChatOpenAI(model="gpt-4o-min", temperature=0)

llm3 = Ollama(model="llama3.2:1b", temperature=0.1)

In [13]:
llm4 = ChatOllama(model="llama3.1", temperature=0)

In [35]:
agent = create_agent(
    model=llm4,
    tools=[],
    system_prompt="You are a recommended assistant"
)

In [36]:
response= agent.invoke(input={"messages":[
    {'role':'user','content':'Berlin is the capital of Germany'}
]})

KeyboardInterrupt: 

In [ ]:
print(response['messages'][-1].content)

[HumanMessage(content='irrelevant', additional_kwargs={}, response_metadata={}, id='2f8b574d-594f-4053-8fd3-549214220aae'), AIMessage(content="It seems like you might be looking for something, but I'm not sure what it is. Could you please provide more context or clarify what you need assistance with? I'll do my best to help!", additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-04-29T18:43:50.066086Z', 'done': True, 'done_reason': 'stop', 'total_duration': 151775206210, 'load_duration': 114447286, 'prompt_eval_count': 23, 'prompt_eval_duration': 30983634741, 'eval_count': 42, 'eval_duration': 120653054557, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019dda8b-a2b9-7870-8a03-cb05dd027051-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 23, 'output_tokens': 42, 'total_tokens': 65})]


In [ ]:
response= agent.invoke(input={"messages":[
    {'role':'user','content':'what is the capital of Germany'}
]})

In [ ]:
print(response['messages'][-1].content)

That's partially correct! Berlin is indeed the capital city of Germany, but it's not the only one. The official capital of Germany is actually a bit more complex.

Since 1990, when East and West Germany were reunified, the capital has been a joint effort between the two former states. However, in 1999, the German parliament (Bundestag) voted to make Berlin the sole capital of Germany, replacing Bonn which had served as the capital since 1949.

So while it's common for people to refer to both Berlin and Bonn as capitals, technically speaking, Berlin is the sole capital of Germany.


we can define two differents model. one for test-enviroment and the other for prod


In [40]:
@wrap_model_call
def dynamic_model_selection(request:ModelRequest, handler) -> ModelResponse :
    env= request.runtime.context.get("env","test")
    if env == "test":
        model = llm4
    else:
        model = llm
    return handler(request.override(model=model))



In [41]:
agent2 = create_agent(
    model=llm4,
    tools=[],
    middleware=[dynamic_model_selection],
    debug=True,
)

In [43]:

response = agent.invoke(input={"messages":[HumanMessage("Mein Name ist Zakaria")]})

In [44]:
print(response['messages'][-1].content)

Hallo Zakaria! Wie kann ich dir heute helfen?


In [37]:
from langgraph.checkpoint.memory import InMemorySaver


In [38]:
memory = InMemorySaver()


In [39]:
agent = create_agent(
    model=llm4,
    tools=[],
    system_prompt="You are a recommended assistant",
    checkpointer=memory
)

config= {"configurable":{"thread_id":1}}

In [18]:
response= agent.invoke(
    input={"messages":[HumanMessage("Mein name ist Zakaria")]},
    config=config,
)

In [19]:
print(response['messages'][-1].content)

Hallo Zakaria! Wie kann ich dir heute helfen?


In [20]:
response= agent.invoke(
    input={"messages":[HumanMessage("Wie ist mein Name")]},
    config=config,
)

In [21]:
print(response['messages'][-1].content)

Dein Name ist Zakaria.


In [24]:
from langchain.tools import tool



In [40]:
@tool()
def get_meteo(city:str):
    """
    Weather of teh given city
    """

    return {
        "city": city,
        "temperature":23,
        "humidity":80,
        "preassure":102,
    }

@tool()
def get_user_info(name:str):
    """
    Infos about the given person
    """

    return {
        "name": name,
        "salary":2300,
        "departement":entwicklung,
    }    

In [53]:
agent_t =create_agent(
    model=llm4,
    tools=[get_meteo, get_user_info],
    system_prompt="Check User questions and use provider tools to answer his question",
    checkpointer=memory
)

In [54]:
config={"configurable":{"thread_id":1}}
response = agent_t.invoke(input={'messages':[HumanMessage("weather in Munich")]}, config=config)

In [55]:
print(response['messages'][-1].content)

The current weather in Munich is:

* Temperature: 23°C (73°F)
* Humidity: 80%
* Pressure: 1020 mbar

Please note that this information is subject to change and might not be up-to-date. For the most accurate and recent weather forecast, I recommend checking a reliable weather website or app.
